# Business Glossary to Cortex Analyst Semantic Model Demo

This notebook demonstrates how to leverage an existing Business Glossary table in Snowflake to enrich your Cortex Analyst semantic models with business-approved descriptions.

## What We'll Cover
1. Create sample source tables (fake banking data)
2. Create and populate a Business Glossary table
3. Query and join glossary with table metadata
4. Generate a semantic model YAML automatically
5. Create a Semantic View programmatically using `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML`

---
## Setup: Connect to Snowflake

In [1]:
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
import pandas as pd
import yaml
import json

# Get the active Snowflake session (works in Snowflake Notebooks)
try:
    session = get_active_session()
except:
    session = Session.builder.create()

print("Connected to Snowflake!")
print(f"Current database: {session.get_current_database()}")
print(f"Current schema: {session.get_current_schema()}")

Connected to Snowflake!
Current database: "DEMO"
Current schema: "PUBLIC"


---
## Step 1: Create Sample Database and Schema

In [2]:
# Create database and set context for our demo
session.sql("CREATE DATABASE IF NOT EXISTS DEMO").collect()
session.sql("USE DATABASE DEMO").collect()
session.sql("USE SCHEMA PUBLIC").collect()
print("Database and schema ready")

Database and schema ready


---
## Step 2: Create Sample Source Tables (Fake Banking Data)

In [3]:
# Create CUSTOMERS table
session.sql("""
CREATE OR REPLACE TABLE DEMO.PUBLIC.CUSTOMERS (
    CUSTOMER_ID VARCHAR(20) PRIMARY KEY,
    FIRST_NAME VARCHAR(50),
    LAST_NAME VARCHAR(50),
    EMAIL VARCHAR(100),
    PHONE_NUMBER VARCHAR(20),
    DATE_OF_BIRTH DATE,
    CUSTOMER_SINCE DATE,
    CUSTOMER_SEGMENT VARCHAR(20),
    CREDIT_SCORE INTEGER,
    ANNUAL_INCOME NUMBER(12,2),
    STATE_CODE VARCHAR(2),
    IS_ACTIVE BOOLEAN
)
""").collect()
print("Created CUSTOMERS table")

Created CUSTOMERS table


In [4]:
# Create ACCOUNTS table
session.sql("""
CREATE OR REPLACE TABLE DEMO.PUBLIC.ACCOUNTS (
    ACCOUNT_ID VARCHAR(20) PRIMARY KEY,
    CUSTOMER_ID VARCHAR(20),
    ACCOUNT_TYPE VARCHAR(20),
    ACCOUNT_STATUS VARCHAR(15),
    OPENED_DATE DATE,
    CLOSED_DATE DATE,
    CURRENT_BALANCE NUMBER(15,2),
    AVAILABLE_BALANCE NUMBER(15,2),
    INTEREST_RATE NUMBER(5,4),
    OVERDRAFT_LIMIT NUMBER(10,2),
    LAST_ACTIVITY_DATE DATE
)
""").collect()
print("Created ACCOUNTS table")

Created ACCOUNTS table


In [5]:
# Create TRANSACTIONS table
session.sql("""
CREATE OR REPLACE TABLE DEMO.PUBLIC.TRANSACTIONS (
    TRANSACTION_ID VARCHAR(30) PRIMARY KEY,
    ACCOUNT_ID VARCHAR(20),
    TRANSACTION_DATE TIMESTAMP_NTZ,
    POSTED_DATE DATE,
    TRANSACTION_TYPE VARCHAR(20),
    TRANSACTION_CATEGORY VARCHAR(30),
    AMOUNT NUMBER(12,2),
    RUNNING_BALANCE NUMBER(15,2),
    MERCHANT_NAME VARCHAR(100),
    MERCHANT_CATEGORY_CODE VARCHAR(4),
    IS_RECURRING BOOLEAN,
    CHANNEL VARCHAR(20)
)
""").collect()
print("Created TRANSACTIONS table")

Created TRANSACTIONS table


---
## Step 3: Insert Fake Data into Source Tables

In [6]:
# Insert fake customers
session.sql("""
INSERT INTO DEMO.PUBLIC.CUSTOMERS VALUES
    ('CUST001', 'John', 'Smith', 'john.smith@email.com', '555-0101', '1985-03-15', '2019-06-01', 'PREMIUM', 750, 95000.00, 'CA', TRUE),
    ('CUST002', 'Sarah', 'Johnson', 'sarah.j@email.com', '555-0102', '1990-07-22', '2020-01-15', 'STANDARD', 680, 62000.00, 'TX', TRUE),
    ('CUST003', 'Michael', 'Williams', 'mwilliams@email.com', '555-0103', '1978-11-08', '2018-03-20', 'PREMIUM', 790, 125000.00, 'NY', TRUE),
    ('CUST004', 'Emily', 'Brown', 'emily.brown@email.com', '555-0104', '1995-01-30', '2021-09-10', 'BASIC', 620, 45000.00, 'FL', TRUE),
    ('CUST005', 'David', 'Jones', 'djones@email.com', '555-0105', '1982-06-18', '2017-11-05', 'PREMIUM', 810, 150000.00, 'WA', TRUE),
    ('CUST006', 'Lisa', 'Garcia', 'lisa.garcia@email.com', '555-0106', '1988-09-25', '2022-02-28', 'STANDARD', 710, 78000.00, 'IL', TRUE),
    ('CUST007', 'Robert', 'Martinez', 'rmartinez@email.com', '555-0107', '1975-04-12', '2016-08-15', 'PREMIUM', 770, 110000.00, 'AZ', TRUE),
    ('CUST008', 'Jennifer', 'Anderson', 'janderson@email.com', '555-0108', '1992-12-03', '2023-01-20', 'BASIC', 590, 38000.00, 'CO', FALSE),
    ('CUST009', 'William', 'Taylor', 'wtaylor@email.com', '555-0109', '1980-08-27', '2019-04-12', 'STANDARD', 700, 72000.00, 'GA', TRUE),
    ('CUST010', 'Amanda', 'Thomas', 'athomas@email.com', '555-0110', '1998-02-14', '2024-03-01', 'BASIC', 650, 52000.00, 'NC', TRUE)
""").collect()
print("Inserted 10 customers")

Inserted 10 customers


In [7]:
# Insert fake accounts
session.sql("""
INSERT INTO DEMO.PUBLIC.ACCOUNTS VALUES
    ('ACC001', 'CUST001', 'CHECKING', 'ACTIVE', '2019-06-01', NULL, 15420.50, 15420.50, 0.0010, 500.00, '2024-12-15'),
    ('ACC002', 'CUST001', 'SAVINGS', 'ACTIVE', '2019-06-15', NULL, 45000.00, 45000.00, 0.0425, 0.00, '2024-12-10'),
    ('ACC003', 'CUST002', 'CHECKING', 'ACTIVE', '2020-01-15', NULL, 3250.75, 3250.75, 0.0005, 200.00, '2024-12-14'),
    ('ACC004', 'CUST002', 'SAVINGS', 'ACTIVE', '2020-02-01', NULL, 12500.00, 12500.00, 0.0400, 0.00, '2024-11-30'),
    ('ACC005', 'CUST003', 'CHECKING', 'ACTIVE', '2018-03-20', NULL, 28750.25, 28750.25, 0.0015, 1000.00, '2024-12-15'),
    ('ACC006', 'CUST003', 'MONEY_MARKET', 'ACTIVE', '2018-04-01', NULL, 150000.00, 150000.00, 0.0475, 0.00, '2024-12-01'),
    ('ACC007', 'CUST004', 'CHECKING', 'ACTIVE', '2021-09-10', NULL, 1875.30, 1375.30, 0.0005, 100.00, '2024-12-13'),
    ('ACC008', 'CUST005', 'CHECKING', 'ACTIVE', '2017-11-05', NULL, 42000.00, 42000.00, 0.0020, 2000.00, '2024-12-15'),
    ('ACC009', 'CUST005', 'CD', 'ACTIVE', '2023-06-01', NULL, 100000.00, 0.00, 0.0500, 0.00, '2023-06-01'),
    ('ACC010', 'CUST006', 'CHECKING', 'ACTIVE', '2022-02-28', NULL, 5620.80, 5620.80, 0.0010, 300.00, '2024-12-12'),
    ('ACC011', 'CUST007', 'SAVINGS', 'ACTIVE', '2016-08-20', NULL, 85000.00, 85000.00, 0.0450, 0.00, '2024-12-05'),
    ('ACC012', 'CUST008', 'CHECKING', 'CLOSED', '2023-01-20', '2024-06-15', 0.00, 0.00, 0.0005, 0.00, '2024-06-15'),
    ('ACC013', 'CUST009', 'CHECKING', 'ACTIVE', '2019-04-12', NULL, 8900.45, 8900.45, 0.0010, 400.00, '2024-12-14'),
    ('ACC014', 'CUST010', 'CHECKING', 'ACTIVE', '2024-03-01', NULL, 2100.00, 2100.00, 0.0005, 100.00, '2024-12-11')
""").collect()
print("Inserted 14 accounts")

Inserted 14 accounts


In [8]:
# Insert fake transactions
session.sql("""
INSERT INTO DEMO.PUBLIC.TRANSACTIONS VALUES
    ('TXN0001', 'ACC001', '2024-12-15 09:30:00', '2024-12-15', 'DEBIT', 'GROCERIES', -125.50, 15420.50, 'Whole Foods Market', '5411', FALSE, 'CARD'),
    ('TXN0002', 'ACC001', '2024-12-14 14:22:00', '2024-12-14', 'DEBIT', 'DINING', -45.00, 15546.00, 'Olive Garden', '5812', FALSE, 'CARD'),
    ('TXN0003', 'ACC001', '2024-12-13 08:00:00', '2024-12-13', 'CREDIT', 'PAYROLL', 3500.00, 15591.00, 'ACME Corp Payroll', '0000', TRUE, 'ACH'),
    ('TXN0004', 'ACC002', '2024-12-10 00:00:00', '2024-12-10', 'CREDIT', 'INTEREST', 159.38, 45000.00, 'Interest Payment', '0000', TRUE, 'INTERNAL'),
    ('TXN0005', 'ACC003', '2024-12-14 16:45:00', '2024-12-14', 'DEBIT', 'UTILITIES', -150.00, 3250.75, 'Electric Company', '4900', TRUE, 'ACH'),
    ('TXN0006', 'ACC003', '2024-12-12 11:30:00', '2024-12-12', 'DEBIT', 'SHOPPING', -89.99, 3400.75, 'Amazon', '5999', FALSE, 'ONLINE'),
    ('TXN0007', 'ACC005', '2024-12-15 10:15:00', '2024-12-15', 'DEBIT', 'TRANSFER', -5000.00, 28750.25, 'Transfer to Savings', '0000', FALSE, 'MOBILE'),
    ('TXN0008', 'ACC005', '2024-12-14 09:00:00', '2024-12-14', 'CREDIT', 'PAYROLL', 8500.00, 33750.25, 'Tech Corp Direct Deposit', '0000', TRUE, 'ACH'),
    ('TXN0009', 'ACC006', '2024-12-01 00:00:00', '2024-12-01', 'CREDIT', 'TRANSFER', 5000.00, 150000.00, 'Transfer from Checking', '0000', FALSE, 'INTERNAL'),
    ('TXN0010', 'ACC007', '2024-12-13 19:20:00', '2024-12-13', 'DEBIT', 'ENTERTAINMENT', -15.99, 1875.30, 'Netflix', '4899', TRUE, 'ONLINE'),
    ('TXN0011', 'ACC007', '2024-12-10 12:00:00', '2024-12-10', 'DEBIT', 'GROCERIES', -67.45, 1891.29, 'Kroger', '5411', FALSE, 'CARD'),
    ('TXN0012', 'ACC008', '2024-12-15 08:45:00', '2024-12-15', 'CREDIT', 'PAYROLL', 6250.00, 42000.00, 'Consulting Inc', '0000', TRUE, 'ACH'),
    ('TXN0013', 'ACC010', '2024-12-12 15:30:00', '2024-12-12', 'DEBIT', 'GAS', -55.00, 5620.80, 'Shell Gas Station', '5541', FALSE, 'CARD'),
    ('TXN0014', 'ACC013', '2024-12-14 13:00:00', '2024-12-14', 'DEBIT', 'HEALTHCARE', -125.00, 8900.45, 'CVS Pharmacy', '5912', FALSE, 'CARD'),
    ('TXN0015', 'ACC014', '2024-12-11 10:00:00', '2024-12-11', 'CREDIT', 'DEPOSIT', 500.00, 2100.00, 'Mobile Deposit', '0000', FALSE, 'MOBILE')
""").collect()
print("Inserted 15 transactions")

Inserted 15 transactions


---
## Step 4: Create the Business Glossary Table & Extract from Document

In [9]:
# Create Business Glossary table
session.sql("""
CREATE OR REPLACE TABLE DEMO.PUBLIC.BUSINESS_GLOSSARY (
    DATABASE VARCHAR(100),
    SCHEMA VARCHAR(100),
    TABLE_NAME VARCHAR(100),
    COLUMN_NAME VARCHAR(100),
    BUSINESS_DESCRIPTION VARCHAR(1000),
    DATA_STEWARD VARCHAR(100),
    LAST_UPDATED DATE,
    SYNONYMS VARCHAR(500)
)
""").collect()
print("Created BUSINESS_GLOSSARY table")

Created BUSINESS_GLOSSARY table


In [10]:
# Drop and recreate stage with server-side encryption (required for AI_EXTRACT)
session.sql("DROP STAGE IF EXISTS DEMO.PUBLIC.BUSINESS_GLOSSARY_DOCS").collect()
session.sql("""
CREATE STAGE DEMO.PUBLIC.BUSINESS_GLOSSARY_DOCS
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
""").collect()
print("Created stage BUSINESS_GLOSSARY_DOCS with server-side encryption")

# Upload the Word doc to the stage
session.file.put(
    "Business_Glossary_Customers.docx",
    "@DEMO.PUBLIC.BUSINESS_GLOSSARY_DOCS",
    auto_compress=False,
    overwrite=True
)
print("Uploaded Business_Glossary_Customers.docx to stage")

# Refresh the stage directory
session.sql("ALTER STAGE DEMO.PUBLIC.BUSINESS_GLOSSARY_DOCS REFRESH").collect()
print("Stage refreshed")

# Verify the file is in the stage
session.sql("LS @DEMO.PUBLIC.BUSINESS_GLOSSARY_DOCS").show()

Created stage BUSINESS_GLOSSARY_DOCS with server-side encryption
Uploaded Business_Glossary_Customers.docx to stage
Stage refreshed
----------------------------------------------------------------------------------------------------------------------------------
|"name"                                              |"size"  |"md5"                             |"last_modified"                |
----------------------------------------------------------------------------------------------------------------------------------
|business_glossary_docs/Business_Glossary_Custom...  |39798   |df75a687f3d4d8c7b3ec9ed26aeb0360  |Fri, 27 Feb 2026 19:52:12 GMT  |
----------------------------------------------------------------------------------------------------------------------------------



In [11]:
# Use AI_EXTRACT to extract glossary entries from the Word document
extract_result = session.sql("""
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO.PUBLIC.BUSINESS_GLOSSARY_DOCS', 'Business_Glossary_Customers.docx'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'glossary_entries': {
                    'type': 'object',
                    'description': 'All rows from the business glossary table',
                    'column_ordering': ['database', 'schema', 'table_name', 'column_name', 'business_description', 'data_steward', 'last_updated', 'synonyms'],
                    'properties': {
                        'database': {'type': 'array', 'description': 'Database name'},
                        'schema': {'type': 'array', 'description': 'Schema name'},
                        'table_name': {'type': 'array', 'description': 'Table name'},
                        'column_name': {'type': 'array', 'description': 'Column name'},
                        'business_description': {'type': 'array', 'description': 'Business description of the column'},
                        'data_steward': {'type': 'array', 'description': 'Data steward responsible'},
                        'last_updated': {'type': 'array', 'description': 'Date last updated'},
                        'synonyms': {'type': 'array', 'description': 'Comma-separated synonyms'}
                    }
                }
            }
        }
    }
) AS result
""").collect()

import json
extracted = json.loads(extract_result[0]['RESULT'])
print(json.dumps(extracted, indent=2))

{
  "error": null,
  "response": {
    "glossary_entries": {
      "business_description": [
        "Unique identifier assigned to each customer at account opening. Format: CUSTOMER_ID",
        "Custom legal first name of the customer as it appears on their government -issued ID",
        "Legal last name (surname) of the customer as it appears on their government -issued ID",
        "Primary email address used for account communications and digital banking access",
        "Primary contact phone number for account alerts and two-factor authentication",
        "Customer date of birth used for identity verification and age -restricted services",
        "Date when the customer first opened an account with the bank",
        "Customer tier based on relationship value: BASIC (< $50K), STANDARD ($50K- $100K), PREMIUM (> $100K in total deposits)",
        "Most recent FICO credit score (300-850) pulled during account review. Updated quarterly.",
        "Self -reported annual household 

In [12]:
# Parse the AI_EXTRACT result and insert into BUSINESS_GLOSSARY table
entries = extracted['response']['glossary_entries']
num_rows = len(entries['database'])
print(f"Extracted {num_rows} glossary entries from document")

# Build INSERT values from the extracted arrays
values = []
for i in range(num_rows):
    row = (
        entries['database'][i],
        entries['schema'][i],
        entries['table_name'][i],
        entries['column_name'][i],
        entries['business_description'][i],
        entries['data_steward'][i],
        entries['last_updated'][i],
        entries['synonyms'][i]
    )
    escaped = tuple(v.replace("'", "''") for v in row)
    values.append(f"('{escaped[0]}', '{escaped[1]}', '{escaped[2]}', '{escaped[3]}', '{escaped[4]}', '{escaped[5]}', '{escaped[6]}', '{escaped[7]}')")

insert_sql = f"""
INSERT INTO DEMO.PUBLIC.BUSINESS_GLOSSARY 
    (DATABASE, SCHEMA, TABLE_NAME, COLUMN_NAME, BUSINESS_DESCRIPTION, DATA_STEWARD, LAST_UPDATED, SYNONYMS)
VALUES
    {', '.join(values)}
"""
session.sql(insert_sql).collect()

# Verify the insert
count = session.sql("SELECT COUNT(*) AS CNT FROM DEMO.PUBLIC.BUSINESS_GLOSSARY").collect()[0]['CNT']
print(f"Inserted {num_rows} rows. Total rows in BUSINESS_GLOSSARY: {count}")

Extracted 36 glossary entries from document
Inserted 36 rows. Total rows in BUSINESS_GLOSSARY: 36


---
## Step 5: Preview the Business Glossary

In [13]:
# Query the glossary
glossary_df = session.sql("""
    SELECT TABLE_NAME, COLUMN_NAME, BUSINESS_DESCRIPTION, SYNONYMS
    FROM DEMO.PUBLIC.BUSINESS_GLOSSARY
    ORDER BY TABLE_NAME, COLUMN_NAME
""").to_pandas()

print(f"Total glossary entries: {len(glossary_df)}")
glossary_df.head(15)

Total glossary entries: 36


,TABLE_NAME,COLUMN_NAME,BUSINESS_DESCRIPTION,SYNONYMS
0,ACCOUNTS,ACCOUNT_ID,Unique identifier for each account. Format: AC...,"account number, acct id"
1,ACCOUNTS,ACCOUNT_STATUS,"(interest -bearing), MONEY_MARKET (high -yield...","status, state"
2,ACCOUNTS,ACCOUNT_TYPE,"Product type: CHECKING (daily transactions ), ...","product type, account product"
3,ACCOUNTS,AVAILABLE_BALANCE,Balance available for withdrawal after holds a...,"available funds, withdrawable balance"
4,ACCOUNTS,CLOSED_DATE,Date the account was closed. NULL if account i...,"close date, termination date"
5,ACCOUNTS,CURRENT_BALANCE,Total balance including pending transactions. ...,"balance, ledger balance, total balance"
6,ACCOUNTS,CUSTOMER_ID,Foreign key linking to the CUSTOMERS table. Id...,"client id, owner id"
7,ACCOUNTS,INTEREST_RATE,Annual Percentage Yield (APY) as a decimal. Ex...,"APY, yield, rate"
8,ACCOUNTS,LAST_ACTIVITY_DATE,Date of most recent debit or credit transactio...,"last transaction date, recent activity"
9,ACCOUNTS,OPENED_DATE,Date the account was officially opened and bec...,"open date, start date, activation date"


---
## Step 6: Join Glossary with Table Metadata

This is the key step - joining the Business Glossary with `INFORMATION_SCHEMA.COLUMNS` to enrich column metadata with business descriptions.

In [14]:
# Query to join glossary with actual table metadata
enriched_df = session.sql("""
SELECT 
    c.TABLE_NAME,
    c.COLUMN_NAME,
    c.DATA_TYPE,
    c.IS_NULLABLE,
    c.ORDINAL_POSITION,
    g.BUSINESS_DESCRIPTION,
    g.SYNONYMS,
    g.DATA_STEWARD
FROM DEMO.INFORMATION_SCHEMA.COLUMNS c
LEFT JOIN DEMO.PUBLIC.BUSINESS_GLOSSARY g
    ON g.DATABASE = c.TABLE_CATALOG
   AND g.SCHEMA = c.TABLE_SCHEMA
   AND g.TABLE_NAME = c.TABLE_NAME
   AND g.COLUMN_NAME = c.COLUMN_NAME
WHERE c.TABLE_SCHEMA = 'PUBLIC'
  AND c.TABLE_NAME IN ('CUSTOMERS', 'ACCOUNTS', 'TRANSACTIONS')
ORDER BY c.TABLE_NAME, c.ORDINAL_POSITION
""").to_pandas()

print(f"Enriched metadata rows: {len(enriched_df)}")
enriched_df

Enriched metadata rows: 35


,TABLE_NAME,COLUMN_NAME,DATA_TYPE,IS_NULLABLE,ORDINAL_POSITION,BUSINESS_DESCRIPTION,SYNONYMS,DATA_STEWARD
0,ACCOUNTS,ACCOUNT_ID,TEXT,NO,1,Unique identifier for each account. Format: AC...,"account number, acct id",Data Governance Team
1,ACCOUNTS,CUSTOMER_ID,TEXT,YES,2,Foreign key linking to the CUSTOMERS table. Id...,"client id, owner id",Data Governance Team
2,ACCOUNTS,ACCOUNT_TYPE,TEXT,YES,3,"Product type: CHECKING (daily transactions ), ...","product type, account product",Product Team
3,ACCOUNTS,ACCOUNT_STATUS,TEXT,YES,4,"(interest -bearing), MONEY_MARKET (high -yield...","status, state",Operations
4,ACCOUNTS,OPENED_DATE,DATE,YES,5,Date the account was officially opened and bec...,"open date, start date, activation date",Operations
5,ACCOUNTS,CLOSED_DATE,DATE,YES,6,Date the account was closed. NULL if account i...,"close date, termination date",Operations
6,ACCOUNTS,CURRENT_BALANCE,NUMBER,YES,7,Total balance including pending transactions. ...,"balance, ledger balance, total balance",Finance
7,ACCOUNTS,AVAILABLE_BALANCE,NUMBER,YES,8,Balance available for withdrawal after holds a...,"available funds, withdrawable balance",Finance
8,ACCOUNTS,INTEREST_RATE,NUMBER,YES,9,Annual Percentage Yield (APY) as a decimal. Ex...,"APY, yield, rate",Product Team
9,ACCOUNTS,OVERDRAFT_LIMIT,NUMBER,YES,10,Maximum overdraft protection amount for checki...,"overdraft protection, OD limit",Risk Management


---
## Step 7: Enhance Existing Semantic View YAML with Business Glossary

Instead of generating a new semantic model from scratch, we'll load the existing `original_sv.yaml` and enrich its column descriptions and synonyms using the Business Glossary data.

In [ ]:
def enhance_semantic_model(semantic_model, glossary_df):
    """Enhance an existing semantic model YAML with Business Glossary descriptions and synonyms.
    
    Args:
        semantic_model: Parsed YAML dict from the existing semantic view
        glossary_df: DataFrame with columns TABLE_NAME, COLUMN_NAME, BUSINESS_DESCRIPTION, SYNONYMS
    
    Returns:
        The enhanced semantic model dict
    """
    # Build a lookup dict from the glossary: (TABLE_NAME, COLUMN_NAME) -> {description, synonyms}
    glossary_lookup = {}
    for _, row in glossary_df.iterrows():
        key = (row['TABLE_NAME'].upper(), row['COLUMN_NAME'].upper())
        glossary_lookup[key] = {
            'description': row.get('BUSINESS_DESCRIPTION'),
            'synonyms': row.get('SYNONYMS')
        }
    
    def parse_synonyms(synonyms_str):
        """Parse comma-separated synonyms into a list."""
        if not synonyms_str or pd.isna(synonyms_str):
            return None
        return [s.strip() for s in str(synonyms_str).split(',') if s.strip()]
    
    # Walk through each table in the semantic model
    for table_def in semantic_model.get('tables', []):
        table_name = table_def.get('name', '').upper()
        
        # Enhance columns in each section: dimensions, time_dimensions, facts/measures
        for section_key in ['dimensions', 'time_dimensions', 'facts', 'measures']:
            columns = table_def.get(section_key, [])
            for col_def in columns:
                col_name = col_def.get('expr', col_def.get('name', '')).upper()
                lookup_key = (table_name, col_name)
                
                if lookup_key in glossary_lookup:
                    entry = glossary_lookup[lookup_key]
                    
                    # Update description if glossary has one
                    if entry['description'] and not pd.isna(entry['description']):
                        col_def['description'] = entry['description']
                    
                    # Add/update synonyms if glossary has them
                    synonyms_list = parse_synonyms(entry['synonyms'])
                    if synonyms_list:
                        col_def['synonyms'] = synonyms_list
    
    return semantic_model


print("Enhancement function defined successfully!")

In [ ]:
# Load the existing semantic view YAML
with open('original_sv.yaml', 'r') as f:
    semantic_model = yaml.safe_load(f)

print(f"Loaded semantic model: '{semantic_model['name']}' with {len(semantic_model['tables'])} tables")

# Enhance it with the Business Glossary (using enriched_df from Step 6)
enhanced_model = enhance_semantic_model(semantic_model, enriched_df)

# Convert to YAML
yaml_output = yaml.dump(enhanced_model, default_flow_style=False, sort_keys=False, allow_unicode=True)
print(yaml_output)

---
## Step 8: Validate the Semantic Model YAML

Before creating the semantic view, we can validate that the YAML specification is correct using `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML` with the `verify_only` parameter set to `TRUE`.

In [17]:
def validate_semantic_model(session, database, schema, yaml_spec):
    """
    Validate a semantic model YAML specification without creating the view.
    
    Args:
        session: Snowpark session
        database: Target database
        schema: Target schema
        yaml_spec: YAML string containing the semantic model specification
    
    Returns:
        Validation result message
    """
    fully_qualified_schema = f"{database}.{schema}"
    
    # Use dollar-quoted string to handle special characters in YAML
    query = f"""CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
        '{fully_qualified_schema}',
        $${yaml_spec}$$,
        TRUE
    )"""
    
    try:
        result = session.sql(query).collect()
        return result[0][0] if result else "No result returned"
    except Exception as e:
        return f"Validation error: {e}"

print("Validation function defined!")

Validation function defined!


In [18]:
# Validate the semantic model before creating
validation_result = validate_semantic_model(
    session=session,
    database='DEMO',
    schema='PUBLIC',
    yaml_spec=yaml_output
)
print(f"Validation result: {validation_result}")

Validation result: YAML file is valid for creating a semantic view. No object has been created yet.


---
## Step 9: Create the Semantic View Programmatically

Now we'll create the actual semantic view using `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML`. This stored procedure creates a semantic view from the YAML specification. The name of the semantic view comes from the `name` field in the YAML.

In [19]:
def create_semantic_view_from_yaml(session, database, schema, yaml_spec):
    """
    Create a semantic view from a YAML specification using SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML.
    
    The semantic view name is taken from the 'name' field in the YAML specification.
    If a semantic view with the same name already exists, it will be replaced.
    
    Args:
        session: Snowpark session
        database: Target database
        schema: Target schema  
        yaml_spec: YAML string containing the semantic model specification
    
    Returns:
        Result message from the stored procedure
    """
    fully_qualified_schema = f"{database}.{schema}"
    
    # Use dollar-quoted string to handle special characters in YAML
    query = f"""CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
        '{fully_qualified_schema}',
        $${yaml_spec}$$
    )"""
    
    try:
        result = session.sql(query).collect()
        return result[0][0] if result else "No result returned"
    except Exception as e:
        return f"Error creating semantic view: {e}"

print("Create function defined!")

Create function defined!


In [20]:
# Create the semantic view
create_result = create_semantic_view_from_yaml(
    session=session,
    database='DEMO',
    schema='PUBLIC',
    yaml_spec=yaml_output
)
print(f"Result: {create_result}")

Result: Semantic view was successfully created.


---
## Step 10: Verify the Semantic View

In [21]:
# List semantic views in the schema
session.sql("SHOW SEMANTIC VIEWS IN SCHEMA DEMO.PUBLIC").to_pandas()

,"""created_on""","""name""","""database_name""","""schema_name""","""comment""","""owner""","""owner_role_type""","""extension"""
0,2026-02-27 11:52:52.621000-08:00,BANKING_SEMANTIC_VIEW,DEMO,PUBLIC,Semantic model for DEMO.PUBLIC banking data wi...,SYSADMIN,ROLE,"[""CA""]"
1,2026-01-22 10:00:43.318000-08:00,BANKING_V1,DEMO,PUBLIC,NaN,SYSADMIN,ROLE,"[""CA""]"
2,2025-12-05 09:56:04.771000-08:00,SEC_FILES,DEMO,PUBLIC,NaN,SYSADMIN,ROLE,"[""CA""]"
3,2025-12-05 13:32:55.407000-08:00,TEST,DEMO,PUBLIC,NaN,SYSADMIN,ROLE,"[""CA""]"
4,2026-01-26 13:29:53.535000-08:00,TEST126,DEMO,PUBLIC,NaN,SYSADMIN,ROLE,"[""CA""]"


In [22]:
# Describe the semantic view
session.sql("DESCRIBE SEMANTIC VIEW DEMO.PUBLIC.BANKING_SEMANTIC_VIEW").to_pandas()

,"""object_kind""","""object_name""","""parent_entity""","""property""","""property_value"""
0,EXTENSION,CA,NaN,VALUE,"{""tables"":[{""name"":""customers"",""dimensions"":[{..."
1,NaN,NaN,NaN,COMMENT,Semantic model for DEMO.PUBLIC banking data wi...
2,TABLE,ACCOUNTS,NaN,BASE_TABLE_DATABASE_NAME,DEMO
3,TABLE,ACCOUNTS,NaN,BASE_TABLE_SCHEMA_NAME,PUBLIC
4,TABLE,ACCOUNTS,NaN,BASE_TABLE_NAME,ACCOUNTS
...,...,...,...,...,...
171,DIMENSION,TRANSACTION_TYPE,TRANSACTIONS,EXPRESSION,TRANSACTION_TYPE
172,DIMENSION,TRANSACTION_TYPE,TRANSACTIONS,DATA_TYPE,VARCHAR(20)
173,DIMENSION,TRANSACTION_TYPE,TRANSACTIONS,SYNONYMS,"[""type"",""txn type"",""debit or credit""]"
174,DIMENSION,TRANSACTION_TYPE,TRANSACTIONS,COMMENT,"Direction of money flow: CREDIT (money in), DE..."


In [23]:
# Read back the YAML from the semantic view (optional)
yaml_from_view = session.sql("""
    SELECT SYSTEM$READ_YAML_FROM_SEMANTIC_VIEW('DEMO.PUBLIC.BANKING_SEMANTIC_VIEW')
""").collect()
print("YAML from semantic view:")
print(yaml_from_view[0][0])

YAML from semantic view:
name: BANKING_SEMANTIC_VIEW
description: Semantic model for DEMO.PUBLIC banking data with business glossary descriptions
tables:
  - name: ACCOUNTS
    description: ACCOUNTS table with enriched business descriptions
    base_table:
      database: DEMO
      schema: PUBLIC
      table: ACCOUNTS
    dimensions:
      - name: ACCOUNT_ID
        synonyms:
          - account number
          - acct id
        description: "Unique identifier for each account. Format: ACCTnnn. One customer may have multiple accounts."
        expr: ACCOUNT_ID
        data_type: VARCHAR(20)
      - name: ACCOUNT_STATUS
        synonyms:
          - status
          - state
        description: "(interest -bearing), MONEY_MARKET (high -yield savings), CD (fixed -term certificate)"
        expr: ACCOUNT_STATUS
        data_type: VARCHAR(15)
      - name: ACCOUNT_TYPE
        synonyms:
          - product type
          - account product
        description: "Product type: CHECKING (dai

---
## Step 11: Test with Cortex Analyst

Now you can use the semantic view with Cortex Analyst to ask natural language questions about your data.

In [24]:
def ask_analyst(session, question, semantic_view='DEMO.PUBLIC.BANKING_SEMANTIC_VIEW'):
    """
    Ask a natural language question using Cortex Analyst with a semantic view.
    
    Args:
        session: Snowpark session
        question: Natural language question
        semantic_view: Fully qualified semantic view name
    
    Returns:
        Response from Cortex Analyst
    """
    # Escape single quotes in the question
    escaped_question = question.replace("'", "''")
    
    query = f"""
    SELECT SNOWFLAKE.CORTEX.ANALYST(
        '{escaped_question}',
        SEMANTIC_VIEW => '{semantic_view}'
    ) as RESPONSE
    """
    
    try:
        result = session.sql(query).collect()
        return result[0]['RESPONSE'] if result else None
    except Exception as e:
        print(f"Error calling Cortex Analyst: {e}")
        return None

print("Analyst function defined!")

Analyst function defined!


In [25]:
# Example usage - uncomment to test
# Note: Requires Cortex Analyst to be enabled in your account

# response = ask_analyst(session, "What is the total balance for each customer segment?")
# print(response)

In [26]:
# Example: Ask about premium customers
# Note: Uncomment and run if you have Cortex Analyst enabled

# response = ask_analyst(session, "How many premium tier customers do we have?")
# print(response)

---
## Step 12: Test Sample Queries

Run some sample queries against the tables to verify the data.

In [27]:
# Test query: Total balance by customer segment
session.sql("""
    SELECT 
        c.CUSTOMER_SEGMENT,
        COUNT(DISTINCT c.CUSTOMER_ID) as CUSTOMER_COUNT,
        SUM(a.CURRENT_BALANCE) as TOTAL_BALANCE
    FROM DEMO.PUBLIC.CUSTOMERS c
    JOIN DEMO.PUBLIC.ACCOUNTS a ON c.CUSTOMER_ID = a.CUSTOMER_ID
    WHERE a.ACCOUNT_STATUS = 'ACTIVE'
    GROUP BY c.CUSTOMER_SEGMENT
    ORDER BY TOTAL_BALANCE DESC
""").to_pandas()

,CUSTOMER_SEGMENT,CUSTOMER_COUNT,TOTAL_BALANCE
0,PREMIUM,4,466170.75
1,STANDARD,3,30272.00
2,BASIC,2,3975.30


In [28]:
# Test query: Transaction categories by volume
session.sql("""
    SELECT 
        TRANSACTION_CATEGORY,
        COUNT(*) as TXN_COUNT,
        SUM(ABS(AMOUNT)) as TOTAL_AMOUNT
    FROM DEMO.PUBLIC.TRANSACTIONS
    GROUP BY TRANSACTION_CATEGORY
    ORDER BY TXN_COUNT DESC
""").to_pandas()

,TRANSACTION_CATEGORY,TXN_COUNT,TOTAL_AMOUNT
0,PAYROLL,3,18250.00
1,GROCERIES,2,192.95
2,TRANSFER,2,10000.00
3,UTILITIES,1,150.00
4,ENTERTAINMENT,1,15.99
5,HEALTHCARE,1,125.00
6,SHOPPING,1,89.99
7,GAS,1,55.00
8,INTEREST,1,159.38
9,DEPOSIT,1,500.00


---
## Step 13: Store YAML in Version Control Table (Optional)

You can also store the YAML in a table for version control and audit purposes.

In [29]:
# Create a table to store semantic model versions
session.sql("""
CREATE OR REPLACE TABLE DEMO.PUBLIC.SEMANTIC_MODEL_VERSIONS (
    MODEL_NAME VARCHAR(100),
    VERSION INTEGER,
    YAML_CONTENT VARCHAR(100000),
    CREATED_BY VARCHAR(100),
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    DESCRIPTION VARCHAR(500)
)
""").collect()
print("Created SEMANTIC_MODEL_VERSIONS table")

Created SEMANTIC_MODEL_VERSIONS table


In [30]:
# Store the semantic model YAML
escaped_yaml = yaml_output.replace("'", "''")

session.sql(f"""
INSERT INTO DEMO.PUBLIC.SEMANTIC_MODEL_VERSIONS (MODEL_NAME, VERSION, YAML_CONTENT, CREATED_BY, DESCRIPTION)
VALUES (
    'BANKING_SEMANTIC_VIEW',
    1,
    '{escaped_yaml}',
    CURRENT_USER(),
    'Initial version generated from Business Glossary'
)
""").collect()

print("Semantic model saved to SEMANTIC_MODEL_VERSIONS table")

Semantic model saved to SEMANTIC_MODEL_VERSIONS table


In [31]:
# View stored versions
session.sql("""
    SELECT MODEL_NAME, VERSION, CREATED_BY, CREATED_AT, DESCRIPTION, LENGTH(YAML_CONTENT) as YAML_LENGTH
    FROM DEMO.PUBLIC.SEMANTIC_MODEL_VERSIONS
    ORDER BY MODEL_NAME, VERSION DESC
""").to_pandas()

,MODEL_NAME,VERSION,CREATED_BY,CREATED_AT,DESCRIPTION,YAML_LENGTH
0,BANKING_SEMANTIC_VIEW,1,CHASE,2026-02-27 11:52:57.292,Initial version generated from Business Glossary,8907


---
## Summary

In this notebook, we:

1. **Created sample banking tables** - CUSTOMERS, ACCOUNTS, TRANSACTIONS with realistic fake data
2. **Built a Business Glossary** - with detailed descriptions, data stewards, and synonyms
3. **Joined glossary with metadata** - using INFORMATION_SCHEMA.COLUMNS
4. **Generated a semantic model** - programmatically with Python
5. **Validated the YAML** - using `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML` with `verify_only=TRUE`
6. **Created the Semantic View** - using `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML`
7. **Stored versions** - saved YAML in a table for version control

### Key Benefits
- **Consistency**: Column descriptions come from a single source of truth
- **Maintainability**: Update the glossary table, regenerate the YAML
- **Synonyms**: Business terms automatically map to technical columns
- **Governance**: Data steward tracking built into the process
- **Automation**: Entire process can be automated and scheduled

### Using the Semantic View
The semantic view `DEMO.PUBLIC.BANKING_SEMANTIC_VIEW` is now ready to use with:
- Cortex Analyst in Snowsight
- The `SNOWFLAKE.CORTEX.ANALYST()` function
- Any application that integrates with Cortex Analyst

---
## Cleanup (Optional)

In [32]:
# Uncomment to clean up demo resources
# session.sql("DROP SEMANTIC VIEW IF EXISTS DEMO.PUBLIC.BANKING_SEMANTIC_VIEW").collect()
# session.sql("DROP TABLE IF EXISTS DEMO.PUBLIC.CUSTOMERS").collect()
# session.sql("DROP TABLE IF EXISTS DEMO.PUBLIC.ACCOUNTS").collect()
# session.sql("DROP TABLE IF EXISTS DEMO.PUBLIC.TRANSACTIONS").collect()
# session.sql("DROP TABLE IF EXISTS DEMO.PUBLIC.BUSINESS_GLOSSARY").collect()
# session.sql("DROP TABLE IF EXISTS DEMO.PUBLIC.SEMANTIC_MODEL_VERSIONS").collect()
# print("Cleaned up demo resources")